# RAG with OpenGenAI Stack - Streaming Responses

This notebook demonstrates **RAG queries using OpenGenAI Stack (OGX)** with **streaming responses**.

**Prerequisites:**
- Run [rag_pipeline_build.ipynb](../build_rag_pipeline/rag_pipeline_build.ipynb) first to deploy the pipeline and ingest data
- OpenGenAI Stack deployed via [Deploying OGX](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.3/html-single/working_with_llama_stack/working_with_llama_stack#deploying-llama-stack-server_rag)
- This notebook must run **inside the cluster** (RHOAI notebook server)

## What This Notebook Covers

1. **OGX Setup** — Connect to the OGX API
2. **Verify Services** — Check Milvus and OGX endpoints
3. **RAG with Streaming** — Real-time streaming responses
4. **Side-by-Side Comparison** — RAG vs. Direct LLM
5. **Interactive Streaming Q&A** — Live query loop with streaming

In [ ]:
# Install required packages
!pip install -q pymilvus sentence-transformers llama-stack-client requests

---
## 1. Configuration

These values must match your deployment.

In [ ]:
NAMESPACE = "ray-docling"

# Milvus
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
COLLECTION_NAME = "open_rag_pdfs"

# Embedding
EMBEDDING_MODE = "local"  # Must match pipeline: "local" or "service"
EMBEDDING_MODEL = "ibm-granite/granite-embedding-125m-english"
EMBEDDING_DIM = 768
EMBEDDING_SERVICE_NAME = "embedding-service"
EMBEDDING_ENDPOINT = (
    f"http://{EMBEDDING_SERVICE_NAME}.{NAMESPACE}.svc.cluster.local:8080"
)

# OpenGenAI Stack (OGX)
OGX_SERVICE = "ogx-custom-distribution"
OGX_URL = f"http://{OGX_SERVICE}.{NAMESPACE}.svc.cluster.local:8321"
MODEL_NAME = "vllm-inference/mistral-7b-instruct-v0-3"  # From ogx-stack.yaml

print(f"Milvus:         {MILVUS_HOST}:{MILVUS_PORT}")
print(f"Collection:     {COLLECTION_NAME}")
print(f"Embedding mode: {EMBEDDING_MODE}")
if EMBEDDING_MODE == "service":
    print(f"Embedding URL:  {EMBEDDING_ENDPOINT}")
else:
    print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"OGX URL:        {OGX_URL}")
print(f"Model:          {MODEL_NAME}")

---
## 2. Verify Services

### 2.1 Milvus Collection

In [ ]:
from pymilvus import Collection, MilvusClient, connections

milvus_uri = f"http://{MILVUS_HOST}:{MILVUS_PORT}"
print(f"Connecting to {milvus_uri}")

milvus_client = MilvusClient(uri=milvus_uri, db_name="default")
connections.connect(alias="default", host=MILVUS_HOST, port=str(MILVUS_PORT))

if not milvus_client.has_collection(COLLECTION_NAME):
    print(f"❌ Collection '{COLLECTION_NAME}' not found.")
    print("Available:", milvus_client.list_collections())
    print("Run the ingestion pipeline first.")
else:
    collection = Collection(COLLECTION_NAME)
    collection.flush()
    milvus_client.load_collection(COLLECTION_NAME)
    print(f"✅ Collection: {COLLECTION_NAME}")
    print(f"   Vectors:    {collection.num_entities}")

In [ ]:
# Sample chunks
samples = milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="chunk_index >= 0",
    output_fields=["source_file", "chunk_index", "text"],
    limit=3,
)

print(f"Sample chunks ({len(samples)}):\n")
for s in samples:
    print(f"  [{s['source_file']} | chunk {s['chunk_index']}]")
    print(f"  {s['text'][:150]}...")
    print()

### 2.2 OGX Endpoint

In [ ]:
import requests

print(f"Checking: {OGX_URL}")

try:
    # Check health/models endpoint
    resp = requests.get(f"{OGX_URL}/v1/models", timeout=10)
    resp.raise_for_status()
    models = resp.json()
    print("\n✅ OGX is running")
    print("Available models:")
    for m in models.get("data", []):
        print(f"  - {m.get('id', m)}")
except requests.ConnectionError:
    print("❌ Connection failed. OGX may not be deployed yet.")
    print("Deploy with: oc apply -f ogx-stack.yaml")
except Exception as e:
    print(f"❌ Error: {e}")

### 2.3 Embedding (mode-dependent)

In [ ]:
if EMBEDDING_MODE == "service":
    print(f"Testing embedding service: {EMBEDDING_ENDPOINT}")
    try:
        resp = requests.post(
            f"{EMBEDDING_ENDPOINT}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": ["test"]},
            timeout=10,
        )
        resp.raise_for_status()
        dim = len(resp.json()["data"][0]["embedding"])
        print(f"✅ Embedding service ready (dim={dim})")
    except Exception as e:
        print(f"❌ Embedding service error: {e}")
else:
    print(f"Loading local model: {EMBEDDING_MODEL}")
    from sentence_transformers import SentenceTransformer

    _test_model = SentenceTransformer(EMBEDDING_MODEL)
    _test_emb = _test_model.encode(["test"], normalize_embeddings=True)
    print(f"✅ Local model ready (dim={_test_emb.shape[1]})")
    del _test_model, _test_emb

---
## 3. Initialize RAG Components with OGX

In [ ]:
import requests
from llama_stack_client import LlamaStackClient

# Embedding setup
embed_model = None
embedding_service_url = None

if EMBEDDING_MODE == "local":
    from sentence_transformers import SentenceTransformer

    embed_model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"Embedding: local ({EMBEDDING_MODEL}, dim={EMBEDDING_DIM})")
else:
    embedding_service_url = EMBEDDING_ENDPOINT
    print(f"Embedding: service ({embedding_service_url})")

# OGX client setup
ogx_client = LlamaStackClient(base_url=OGX_URL)
print(f"OGX: {OGX_URL}")
print("\n✅ RAG components initialized")

### 3.1 Define RAG Functions

In [ ]:
def get_embedding(text):
    """Generate embedding using the configured mode."""
    if EMBEDDING_MODE == "local":
        return embed_model.encode([text], normalize_embeddings=True).tolist()[0]
    else:
        resp = requests.post(
            f"{embedding_service_url}/v1/embeddings",
            json={"model": EMBEDDING_MODEL, "input": [text]},
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()["data"][0]["embedding"]


def search_similar_chunks(question, top_k=5):
    """Embed the question and search Milvus for similar chunks."""
    query_embedding = get_embedding(question)

    results = milvus_client.search(
        collection_name=COLLECTION_NAME,
        data=[query_embedding],
        limit=top_k,
        output_fields=["source_file", "chunk_index", "text"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
    )

    chunks = []
    for hits in results:
        for hit in hits:
            chunks.append({
                "text": hit["entity"]["text"],
                "source_file": hit["entity"]["source_file"],
                "chunk_index": hit["entity"]["chunk_index"],
                "score": hit["distance"],
            })
    return chunks


def build_rag_prompt(question, chunks):
    """Build a prompt with retrieved context for the LLM."""
    context_block = "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"similarity: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )
    return (
        "You are a helpful assistant. Answer the user's question based on the "
        "provided context documents. If the answer is not in the context, say so.\n\n"
        f"## Context\n\n{context_block}\n\n"
        f"## Question\n\n{question}\n\n"
        "## Answer\n\n"
    )


print("✅ RAG functions ready")

---
## 4. RAG with Streaming Responses

**Key Feature:** Real-time streaming output using OGX

In [ ]:
def rag_query_streaming(question, top_k=5, max_tokens=512, temperature=0.1):
    """RAG with streaming: retrieve context -> build prompt -> stream answer."""
    # Step 1: Retrieve similar chunks
    print("🔍 Searching for relevant chunks...")
    chunks = search_similar_chunks(question, top_k=top_k)
    print(f"✅ Found {len(chunks)} relevant chunks\n")

    # Step 2: Build prompt
    prompt = build_rag_prompt(question, chunks)

    # Step 3: Stream response from OGX
    print("💬 Answer (streaming):\n")
    print("-" * 60)

    full_response = ""

    response = ogx_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
        stream=True,  # Enable streaming
    )

    for chunk in response:
        if hasattr(chunk.choices[0], "delta") and hasattr(
            chunk.choices[0].delta, "content"
        ):
            content = chunk.choices[0].delta.content
            if content:
                print(content, end="", flush=True)
                full_response += content

    print("\n" + "-" * 60)

    # Step 4: Show sources
    print(f"\n📚 Sources ({len(chunks)}):")
    for i, c in enumerate(chunks, 1):
        print(
            f"  {i}. {c['source_file']} (chunk {c['chunk_index']}, similarity: {c['score']:.3f})"
        )

    return {
        "question": question,
        "answer": full_response,
        "sources": chunks,
    }


print("✅ Streaming RAG function ready")

### 4.1 Test Streaming RAG Query

In [ ]:
question = "What are the main topics covered in the documents?"

result = rag_query_streaming(question)

### 4.2 Another Streaming Query

In [ ]:
result = rag_query_streaming("Summarize the key findings from the documents.")

---
## 5. Side-by-Side Comparison

Compare **RAG with context** vs. **Direct LLM** (no context)

In [ ]:
question = "What are the main topics covered in the documents?"

print("=" * 80)
print("RAG WITH CONTEXT (Streaming)")
print("=" * 80)
rag_result = rag_query_streaming(question)

print("\n" + "=" * 80)
print("DIRECT LLM WITHOUT CONTEXT (Streaming)")
print("=" * 80)
print("💬 Answer (streaming):\n")
print("-" * 60)

direct_response = ogx_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": question}],
    max_tokens=512,
    temperature=0.1,
    stream=True,
)

for chunk in direct_response:
    if hasattr(chunk.choices[0], "delta") and hasattr(
        chunk.choices[0].delta, "content"
    ):
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)

print("\n" + "-" * 60)

print("\n" + "=" * 80)
print("OBSERVATION")
print("=" * 80)
print("The RAG answer is grounded in actual document content,")
print("while the direct answer is generic and speculative.")

---
## 6. Interactive Streaming Q&A

Ask questions and see answers stream in real-time. Type `quit` to exit.

In [ ]:
print("\n🚀 Interactive RAG Query Session (Streaming Enabled)")
print("Type your questions and watch answers stream in real-time!")
print("Type 'quit', 'exit', or 'q' to stop.\n")

while True:
    question = input("\n❓ Your question: ").strip()
    if question.lower() in ("quit", "exit", "q"):
        print("\n👋 Goodbye!")
        break
    if not question:
        continue

    try:
        rag_query_streaming(question)
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("Please try again.")

---
## 7. Advanced: Inspect Retrieved Chunks

In [ ]:
# Inspect what chunks are being retrieved
question = "What are the main topics?"
chunks = search_similar_chunks(question, top_k=5)

print(f"Top {len(chunks)} chunks for: '{question}'\n")
print("=" * 80)

for i, c in enumerate(chunks, 1):
    print(f"\n📄 Chunk {i} (similarity: {c['score']:.3f})")
    print(f"Source: {c['source_file']}, chunk_index: {c['chunk_index']}")
    print("-" * 80)
    print(c["text"][:400])
    if len(c["text"]) > 400:
        print("[...truncated...]")
    print()

---
## 8. Benefits of Using OpenGenAI Stack

✅ **Unified API** - Single interface for multiple models and providers  
✅ **Streaming Support** - Built-in streaming for real-time responses  
✅ **Session Management** - OGX can manage conversation history via PostgreSQL  
✅ **Observability** - Built-in logging and monitoring capabilities  
✅ **Standardization** - OpenAI-compatible API makes integration easier  
✅ **Extensibility** - Easy to add new models or swap providers  

### Next Steps

1. **Add Memory** - Use OGX session management for multi-turn conversations
2. **Add Tools** - Integrate function calling for dynamic data retrieval
3. **Add Routing** - Use multiple models for different query types
4. **Add Monitoring** - Track usage, latency, and quality metrics

---
## 9. Cleanup

Uncomment to clean up resources.

In [ ]:
# Close Milvus connection
# connections.disconnect("default")
# print("Disconnected from Milvus")